In [10]:
import json, time
import numpy as np
import ollama

In [11]:
import ollama
client = ollama.Client(host="http://127.0.0.1:11434", trust_env=False)
ollama.embed = client.embed
ollama.chat = client.chat

In [12]:
with open("../data/chunks_full.json", encoding="utf-8") as f:
    chunks = json.load(f)
print(len(chunks))


t0 = time.time()
r = ollama.embed(model="bge-m3", input=[c["text"] for c in chunks[:32]])
print(len(r["embeddings"]), len(r["embeddings"][0]), round(time.time() - t0, 1))

6705
32 1024 0.8


In [13]:
texts = [c["text"] for c in chunks]
vecs = []
t0 = time.time()
for i in range(0, len(texts), 32):
    r = ollama.embed(model="bge-m3", input=texts[i:i+32])
    vecs.extend(r["embeddings"])
    if (i//32) % 10 == 0:
        print(f"i: {i} | {round(time.time()-t0, 1)}")

i: 0 | 0.8
i: 320 | 10.2
i: 640 | 22.7
i: 960 | 36.1
i: 1280 | 49.9
i: 1600 | 63.6
i: 1920 | 77.5
i: 2240 | 90.0
i: 2560 | 96.0
i: 2880 | 105.3
i: 3200 | 116.8
i: 3520 | 128.6
i: 3840 | 139.0
i: 4160 | 151.2
i: 4480 | 164.9
i: 4800 | 174.9
i: 5120 | 190.8
i: 5440 | 206.8
i: 5760 | 221.7
i: 6080 | 237.2
i: 6400 | 250.1


In [14]:
emb = np.array(vecs, dtype="float32")
emb = emb / np.linalg.norm(emb, axis=1, keepdims=True)
np.save("../data/emb.npy", emb)
print(emb.shape)                         
print(np.linalg.norm(emb[:3], axis=1)) 

(6705, 1024)
[1. 1. 1.]


In [15]:
def search(query , k=5):
    q = ollama.embed(model="bge-m3", input=query)["embeddings"][0]
    q = np.array(q, dtype="float32")
    q = q / np.linalg.norm(q, keepdims=True)
    scores = emb @ q
    top = np.argsort(scores)[::-1][:k]
    return [(float(scores[i]), chunks[i]) for i in top]

def show(query, k=5):
    print("Q:", query)
    for s, c in search(query, k):
        print(f"  {s:.3f}  [{c['kind']}]  {c['title'][:80]}")

for q in [
    "how does DataLoader num_workers work",
    "как сохранить модель",
    "what is torch.nn.Linear",
    "как ускорить обучение на нескольких GPU",
    "torch.nn.functional.conv2d",
    "how to bake a chocolate cake",
]:
    show(q, 3)

Q: how does DataLoader num_workers work
  0.650  [text]  torch.utils.data > Single- and Multi-process Data Loading > Multi-process data l
  0.648  [text]  Reproducibility > DataLoader
  0.637  [api]  torch.utils.data.DataLoader
Q: как сохранить модель
  0.601  [api]  torch.ao.quantization.quantize_fx.attach_preserved_attrs_to_model
  0.576  [api]  torch.save
  0.562  [api]  torch.distributed.checkpoint.state_dict_saver.save
Q: what is torch.nn.Linear
  0.597  [api]  torch.ao.nn.qat.Linear
  0.593  [api]  torch.ao.nn.qat.dynamic.Linear
  0.592  [api]  torch.nn.utils.fuse_linear_bn_eval
Q: как ускорить обучение на нескольких GPU
  0.641  [text]  Modules > Advanced Features > Distributed Training
  0.627  [api]  torch.cuda.tunable.mgpu_tune_gemm_in_file
  0.626  [text]  Automatic Mixed Precision examples > Working with Multiple GPUs > DistributedDat
Q: torch.nn.functional.conv2d
  0.662  [api]  torch.ao.nn.qat.Conv2d
  0.645  [api]  torch.ao.nn.intrinsic.quantized.modules.conv_add.ConvAdd

In [16]:
res = search("what is torch.nn.Linear", k=len(chunks))
print([i for i, (s, c) in enumerate(res) if c["title"] == "torch.nn.Linear"])

[19]


In [17]:
def find_titles(sub):
    return [c["title"] for c in chunks if sub.lower() in c["title"].lower()]

print(find_titles("DataLoader"))

['Reproducibility > DataLoader', 'torch.utils.data.DataLoader', 'torch.utils.data.DataLoader', 'torch.utils.data.DataLoader', 'torch.utils.data.DataLoader']


In [18]:
def is_hit(title, gold):
    for g in gold:
        if (title == g or title.startswith(g + " >")
                or title.endswith(" > " + g) or (" > " + g + " >") in title):
            return True
    return False

In [19]:
from collections import defaultdict

with open("../data/eval.json", encoding="utf-8") as f:
    items = json.load(f)

def metrics(rows):
    hit = sum(r["rank"] is not None for r in rows) / len(rows)
    mrr = sum(1 / r["rank"] if r["rank"] else 0 for r in rows) / len(rows)
    return round(hit, 2), round(mrr, 2)

def evaluate(search_fn, items, k=5):
    rows = []
    none_scores = []
    for it in items:
        res = search_fn(it["q"], k)
        top_score = res[0][0]
        if it["gold"]:
            rank = next((i+1 for i, (s, c) in enumerate(res) if is_hit(c["title"], it["gold"])), None)
            rows.append({**it, "rank": rank, "top_score": top_score})
        else:
            none_scores.append(top_score)
    return rows, none_scores

def report(rows, none_scores):
    print("всего    ", len(rows), "recall@k, MRR =", metrics(rows))
    for key in ("type", "lang"):
        groups = defaultdict(list)
        for r in rows:
            groups[r[key]].append(r)
        for name, g in groups.items():
            print(f"{key}={name:8}", len(g), metrics(g))
    real = [r["top_score"] for r in rows]
    print("score первого результата, вопросы с ответом: средний",
          round(sum(real) / len(real), 3), "минимум", round(min(real), 3))
    print("score первого результата, вопросы без ответа: средний",
          round(sum(none_scores) / len(none_scores), 3), "максимум", round(max(none_scores), 3))


rows, none_scores = evaluate(search, items, k=5)
print(len(rows), len(none_scores))    # ожидание: 34 8
report(rows, none_scores)
        

34 14
всего     34 recall@k, MRR = (0.91, 0.74)
type=api      10 (0.8, 0.63)
type=howto    12 (0.92, 0.68)
type=concept  12 (1.0, 0.88)
lang=en       20 (0.85, 0.69)
lang=ru       14 (1.0, 0.8)
score первого результата, вопросы с ответом: средний 0.641 минимум 0.551
score первого результата, вопросы без ответа: средний 0.506 максимум 0.686


In [20]:
for r in rows:
    if r["rank"] is None:
        print("Q:", r["q"], "| gold:", r["gold"])
        for s, c in search(r["q"], 3):
            print(f"    {s:.3f}  {c['title'][:90]}")

Q: what does torch.nn.Dropout do | gold: ['torch.nn.Dropout']
    0.673  torch.nn.functional.alpha_dropout
    0.625  torch.nn.AlphaDropout
    0.613  torch.fx > Leaf Modules > Miscellanea
Q: what is torch.nn.Linear | gold: ['torch.nn.Linear']
    0.597  torch.ao.nn.qat.Linear
    0.593  torch.ao.nn.qat.dynamic.Linear
    0.592  torch.nn.utils.fuse_linear_bn_eval
Q: how to save and load a model | gold: ['torch.save', 'Saving and loading torch.nn.Modules']
    0.604  torch.hub.set_dir
    0.597  torch.hub > Loading models from Hub > Running a loaded model:
    0.587  torch.ao.quantization.observer.load_observer_state_dict


In [21]:
%pip install rank_bm25

Note: you may need to restart the kernel to use updated packages.


In [22]:
from rank_bm25 import BM25Okapi
import re

def tok(s):
    return re.findall(r"\w+", s.lower())

bm25 = BM25Okapi([tok(c["text"]) for c in chunks])


def search_bm25(query, k=5):
    s = bm25.get_scores(tok(query))
    top = np.argsort(s)[::-1][:k]
    return [(float(s[i]), chunks[i]) for i in top]


def dense_scores(query):
    q = np.array(ollama.embed(model="bge-m3", input=query)["embeddings"][0], dtype="float32")
    return emb @ (q / np.linalg.norm(q))


def search_hybrid(query, k=5, pool=50, c=60):
    d = dense_scores(query)
    b = bm25.get_scores(tok(query))
    fused = defaultdict(float)
    for scores in (d, b):
        for rank, i in enumerate(np.argsort(scores)[::-1][:pool]):
            fused[i] += 1 / (c + rank + 1)
    top = sorted(fused, key=fused.get, reverse=True)[:k]
    return [(float(d[i]), chunks[i]) for i in top]

In [23]:
for name, fn in [("dense", search), ("bm25", search_bm25), ("hybrid", search_hybrid)]:
    print("=====", name)
    r, n = evaluate(fn, items, k=5)
    report(r, n)

===== dense
всего     34 recall@k, MRR = (0.91, 0.74)
type=api      10 (0.8, 0.63)
type=howto    12 (0.92, 0.68)
type=concept  12 (1.0, 0.88)
lang=en       20 (0.85, 0.69)
lang=ru       14 (1.0, 0.8)
score первого результата, вопросы с ответом: средний 0.641 минимум 0.551
score первого результата, вопросы без ответа: средний 0.506 максимум 0.686
===== bm25
всего     34 recall@k, MRR = (0.53, 0.33)
type=api      10 (0.5, 0.33)
type=howto    12 (0.42, 0.27)
type=concept  12 (0.67, 0.4)
lang=en       20 (0.7, 0.42)
lang=ru       14 (0.29, 0.21)
score первого результата, вопросы с ответом: средний 15.977 минимум 0.0
score первого результата, вопросы без ответа: средний 11.534 максимум 24.763
===== hybrid
всего     34 recall@k, MRR = (0.88, 0.61)
type=api      10 (0.7, 0.45)
type=howto    12 (0.92, 0.59)
type=concept  12 (1.0, 0.76)
lang=en       20 (0.8, 0.6)
lang=ru       14 (1.0, 0.62)
score первого результата, вопросы с ответом: средний 0.629 минимум 0.515
score первого результата, вопр

In [24]:
r, n = evaluate(search, items, k=5)
for x in r:
    if x["rank"] is None:
        print("Q:", x["q"], "| gold:", x["gold"])
        for s, c in search(x["q"], 3):
            print(f"    {s:.3f}  {c['title'][:90]}")

Q: what does torch.nn.Dropout do | gold: ['torch.nn.Dropout']
    0.673  torch.nn.functional.alpha_dropout
    0.625  torch.nn.AlphaDropout
    0.613  torch.fx > Leaf Modules > Miscellanea
Q: what is torch.nn.Linear | gold: ['torch.nn.Linear']
    0.597  torch.ao.nn.qat.Linear
    0.593  torch.ao.nn.qat.dynamic.Linear
    0.592  torch.nn.utils.fuse_linear_bn_eval
Q: how to save and load a model | gold: ['torch.save', 'Saving and loading torch.nn.Modules']
    0.604  torch.hub.set_dir
    0.597  torch.hub > Loading models from Hub > Running a loaded model:
    0.587  torch.ao.quantization.observer.load_observer_state_dict


In [25]:
def search_hybrid(query, k=5, pool=50, c=60, w_bm25=1.0):
    d = dense_scores(query)
    b = bm25.get_scores(tok(query))
    fused = defaultdict(float)
    for scores, w in ((d, 1.0), (b, w_bm25)):
        for rank, i in enumerate(np.argsort(scores)[::-1][:pool]):
            fused[i] += w / (c + rank + 1)
    top = sorted(fused, key=fused.get, reverse=True)[:k]
    return [(float(d[i]), chunks[i]) for i in top]

for w in (0.0, 0.3, 0.5, 1.0):
    r, n = evaluate(lambda q, k: search_hybrid(q, k, w_bm25=w), items, k=5)
    print(w, metrics(r))

0.0 (0.91, 0.74)
0.3 (0.88, 0.68)
0.5 (0.88, 0.65)
1.0 (0.88, 0.61)


In [26]:
%pip install sentence-transformers

Note: you may need to restart the kernel to use updated packages.


In [27]:
from sentence_transformers import CrossEncoder
reranker = CrossEncoder("BAAI/bge-reranker-v2-m3", device="mps", max_length=512)

def search_rerank(query, k=5, pool=30):
    cand = [c for s, c in search_hybrid(query, pool)]
    scores = reranker.predict([(query, c["text"]) for c in cand])
    order = np.argsort(scores)[::-1][:k]
    return [(float(scores[i]), cand[i]) for i in order]

def search_rerank(query, k=5, pool=30, w_bm25=0.0):
    cand = [c for s, c in search_hybrid(query, pool, w_bm25=w_bm25)]
    scores = reranker.predict([(query, c["text"]) for c in cand])
    order = np.argsort(scores)[::-1][:k]
    return [(float(scores[i]), cand[i]) for i in order]


for name, fn in [("dense", search),
                 ("hybrid", search_hybrid),
                 ("dense+rerank", search_rerank),
                 ("hybrid+rerank", lambda q, k: search_rerank(q, k, w_bm25=1.0))]:
    print("=====", name)
    r, n = evaluate(fn, items, k=5)
    report(r, n)

r, n = evaluate(search_rerank, items, k=5)
real = sorted(x["top_score"] for x in r)
print("с ответом, 5 самых низких:", [round(s, 3) for s in real[:5]])
print("без ответа, 5 самых высоких:", [round(s, 3) for s in sorted(n)[-5:]])
for thr in (0.05, 0.1, 0.2, 0.3):
    kept = sum(x["top_score"] >= thr for x in r) / len(r)
    refused = sum(s < thr for s in n) / len(n)
    print(thr, "вопросов с ответом сохранено:", round(kept, 2), "| вопросов без ответа отклонено:", round(refused, 2))

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

===== dense
всего     34 recall@k, MRR = (0.91, 0.74)
type=api      10 (0.8, 0.63)
type=howto    12 (0.92, 0.68)
type=concept  12 (1.0, 0.88)
lang=en       20 (0.85, 0.69)
lang=ru       14 (1.0, 0.8)
score первого результата, вопросы с ответом: средний 0.641 минимум 0.551
score первого результата, вопросы без ответа: средний 0.506 максимум 0.686
===== hybrid
всего     34 recall@k, MRR = (0.88, 0.61)
type=api      10 (0.7, 0.45)
type=howto    12 (0.92, 0.59)
type=concept  12 (1.0, 0.76)
lang=en       20 (0.8, 0.6)
lang=ru       14 (1.0, 0.62)
score первого результата, вопросы с ответом: средний 0.629 минимум 0.515
score первого результата, вопросы без ответа: средний 0.492 максимум 0.686
===== dense+rerank
всего     34 recall@k, MRR = (0.88, 0.65)
type=api      10 (0.9, 0.53)
type=howto    12 (0.83, 0.61)
type=concept  12 (0.92, 0.78)
lang=en       20 (0.9, 0.66)
lang=ru       14 (0.86, 0.63)
score первого результата, вопросы с ответом: средний 0.859 минимум 0.199
score первого результа

In [28]:
none_q = [it["q"] for it in items if not it["gold"]]
for q in none_q:
    res = search_rerank(q, 1)
    print(round(res[0][0], 3), "|", q, "->", res[0][1]["title"][:70])

0.005 | how to bake a chocolate cake -> torch.sparse.sampled_addmm
0.001 | как приготовить борщ -> torch.sparse.sampled_addmm
0.004 | who won the 2018 world cup -> PyTorch Governance | Maintainers > Core Module maintainers > Distribut
0.0 | какая столица Франции -> Rendezvous
0.045 | how to train a model with tf.keras Model.fit -> Getting Started on Intel GPU > Examples > Training Examples > Train wi
0.006 | how to set up an Nginx reverse proxy -> Operator Registration > Advanced > Custom Operators > Forward Only
0.012 | как настроить Docker compose -> Custom Backends > Registering Custom Backends
0.0 | what is the weather in Paris -> torch.testing.assert_close
0.852 | what does torch.nn.SuperLinear do -> Modules > A Simple Custom Module
0.559 | how to use torch.quantum_compute -> torch.kthvalue
0.133 | how to train a model in TensorFlow with tf.GradientTape -> Getting Started on Intel GPU > Examples > Training Examples > Train wi
0.987 | как использовать torch.magic_optimizer -> torch

In [29]:
corpus_lower = "\n".join(c["text"] for c in chunks).lower()
known_titles = {c["title"].lower() for c in chunks}


def unknown_names(query):
    names = re.findall(r"torch(?:\.\w+)+", query)
    return [n for n in names
            if n.lower() not in known_titles and n.lower() not in corpus_lower]

In [30]:
for it in items:
    u = unknown_names(it["q"])
    if u or not it["gold"]:
        print(("НЕТ gold" if not it["gold"] else "С gold  "), "|", u, "|", it["q"])

НЕТ gold | [] | how to bake a chocolate cake
НЕТ gold | [] | как приготовить борщ
НЕТ gold | [] | who won the 2018 world cup
НЕТ gold | [] | какая столица Франции
НЕТ gold | [] | how to train a model with tf.keras Model.fit
НЕТ gold | [] | how to set up an Nginx reverse proxy
НЕТ gold | [] | как настроить Docker compose
НЕТ gold | [] | what is the weather in Paris
НЕТ gold | ['torch.nn.SuperLinear'] | what does torch.nn.SuperLinear do
НЕТ gold | ['torch.quantum_compute'] | how to use torch.quantum_compute
НЕТ gold | [] | how to train a model in TensorFlow with tf.GradientTape
НЕТ gold | ['torch.magic_optimizer'] | как использовать torch.magic_optimizer
НЕТ gold | [] | how to fine-tune GPT-4 with the OpenAI API
НЕТ gold | [] | как сделать запрос к базе данных PostgreSQL через SQLAlchemy


In [58]:
from collections import defaultdict

title_index = defaultdict(list)
for c in chunks:
    title_index[c["title"].lower()].append(c)


def named_chunks(query, per_name=2):
    out = []
    for n in re.findall(r"torch(?:\.\w+)+", query):
        out += title_index.get(n.lower(), [])[:per_name]
    return out


THRESH = 0.15


def retrieve(query, k=5, gate_pool=30):
    bad = unknown_names(query)
    if bad:
        return f"в документации нет имени {', '.join(bad)}", []
    pinned = named_chunks(query)
    wide = search(query, gate_pool)
    if not pinned:
        scores = reranker.predict([(query, c["text"]) for s, c in wide])                                       
        if max(scores) < THRESH:
            return "в документации не нашлось подходящего раздела", []
    res = [(1.0, c) for c in pinned]
    ids = {c["id"] for c in pinned}
    for s, c in wide:
        if len(res) >= k:
            break
        if c["id"] in ids:
            continue
        res.append((s, c))
        ids.add(c["id"])
    return None, res


# kept_ok, refused_ok = 0, 0
# for it in items:
#     reason, res = retrieve(it["q"])
#     refused = reason is not None
#     if it["gold"]:
#         kept_ok += not refused
#         if refused:
#             print("ЛИШНИЙ ОТКАЗ:", it["q"], "|", reason)
#     else:
#         refused_ok += refused
#         if not refused:
#             print("НЕ ОТКАЗАЛ:", it["q"])

# n_ans = sum(1 for it in items if it["gold"])
# n_none = len(items) - n_ans
# print("с ответом сохранено:", kept_ok, "из", n_ans)
# print("без ответа отклонено:", refused_ok, "из", n_none)

In [59]:
# 1. привязка работает на известных промахах
for q in ["what does torch.nn.Dropout do", "what is torch.nn.Linear"]:
    reason, res = retrieve(q)
    print(q, "->", reason)
    for s, c in res:
        print(f"    {s:.3f}  {c['title'][:70]}")

# 2. тип api
api = [it for it in items if it["type"] == "api"]
hit = 0
for it in api:
    reason, res = retrieve(it["q"])
    hit += any(is_hit(c["title"], it["gold"]) for s, c in res)
print("api gold в контексте:", hit, "из", len(api))

# 3. общий результат и отказы (не должны ухудшиться)
kept = sum(any(is_hit(c["title"], it["gold"]) for s, c in retrieve(it["q"])[1]) for it in items if it["gold"])
refused = sum(retrieve(it["q"])[0] is not None for it in items if not it["gold"])
print("gold в контексте:", kept, "из 34 | отказов:", refused, "из 14")

what does torch.nn.Dropout do -> None
    1.000  torch.nn.Dropout
    0.673  torch.nn.functional.alpha_dropout
    0.625  torch.nn.AlphaDropout
    0.613  torch.fx > Leaf Modules > Miscellanea
    0.607  torch.nn.functional.dropout3d
what is torch.nn.Linear -> None
    1.000  torch.nn.Linear
    0.597  torch.ao.nn.qat.Linear
    0.593  torch.ao.nn.qat.dynamic.Linear
    0.592  torch.nn.utils.fuse_linear_bn_eval
    0.586  torch.nn.functional.linear
api gold в контексте: 10 из 10
gold в контексте: 33 из 34 | отказов: 14 из 14


In [56]:
for it in items:
    if it["gold"]:
        reason, res = retrieve(it["q"])
        if reason:
            print("ОТКЛОНЁН:", it["q"], "|", reason)

In [57]:
kept, sizes = 0, []
for it in items:
    if not it["gold"]:
        continue
    reason, res = retrieve(it["q"])
    sizes.append(len(res))
    kept += any(is_hit(c["title"], it["gold"]) for s, c in res)
print("gold в контексте:", kept, "из", len(sizes), "| фрагментов в среднем:", round(sum(sizes) / len(sizes), 1))

for it in items:
    if not it["gold"]:
        reason, res = retrieve(it["q"])
        print("отказ" if reason else "ОТВЕТИЛ", "|", it["q"])

gold в контексте: 31 из 34 | фрагментов в среднем: 5.0
отказ | how to bake a chocolate cake
отказ | как приготовить борщ
отказ | who won the 2018 world cup
отказ | какая столица Франции
отказ | how to train a model with tf.keras Model.fit
отказ | how to set up an Nginx reverse proxy
отказ | как настроить Docker compose
отказ | what is the weather in Paris
отказ | what does torch.nn.SuperLinear do
отказ | how to use torch.quantum_compute
отказ | how to train a model in TensorFlow with tf.GradientTape
отказ | как использовать torch.magic_optimizer
отказ | how to fine-tune GPT-4 with the OpenAI API
отказ | как сделать запрос к базе данных PostgreSQL через SQLAlchemy


In [32]:
with open("../data/eval_holdout.json", encoding="utf-8") as f:
    holdout = json.load(f)

kept, refused, errors = 0, 0, []
for it in holdout:
    reason, res = retrieve(it["q"])
    if it["gold"]:
        if reason:
            errors.append(("ЛИШНИЙ ОТКАЗ", it["q"], reason))
        else:
            kept += 1
            hit = any(is_hit(c["title"], it["gold"]) for s, c in res)
            if not hit:
                errors.append(("НЕ НАШЁЛ", it["q"], [c["title"][:50] for s, c in res[:3]]))
    else:
        if reason:
            refused += 1
        else:
            errors.append(("НЕ ОТКАЗАЛ", it["q"], res[0][1]["title"][:60]))

print("с ответом, не отклонено:", kept, "из 5")
print("без ответа, отклонено:", refused, "из 5")
for e in errors:
    print(e)

с ответом, не отклонено: 5 из 5
без ответа, отклонено: 5 из 5
('НЕ НАШЁЛ', 'как перенести тензор на GPU', ['Tensor Attributes > torch.device', 'CUDA semantics', 'torch.distributed.send_object_list'])


In [44]:
GEN_MODEL = "qwen2.5:7b"

SYSTEM = (
    "You are a PyTorch documentation assistant. Answer ONLY based on the provided excerpts."
    "If the answer is not contained within them, state that. Reference the excerpts using their numbers, e.g., [1], [2]."
    "Use code examples from the excerpts; do not invent APIs."
)

def answer_language(query):
    return "Russian" if re.search(r"[а-яА-Я]", query) else "English"

def build_context(res, max_chars=3000):
    parts = []
    for i, (score, c) in enumerate(res, 1):
        s = "[" + str(i) + "] " + c["text"][:max_chars]
        parts.append(s)
    return "\n\n".join(parts)

def answer(query, k=5):
    reason, res = retrieve(query, k)
    if reason:
        msg = f"Простите, я не могу ответить на этот вопрос, так как {reason}"
        print(msg)
        return msg, []
    context = build_context(res)
    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": f"Фрагменты:\n{context}\n\nВопрос: {query}\n\nAnswer in {answer_language(query)}"}
    ]
    text = ""
    for part in ollama.chat(model=GEN_MODEL, messages=messages, stream=True, options={"temperature":0.1, "num_ctx": 8192}):
        piece = part["message"]["content"]
        print(piece, end="", flush=True)
        text += piece
    sources = [(c["title"], c["file"]) for s, c in res]
    return text, sources

In [42]:
q = "how does DataLoader num_workers work"
text, sources = answer(q)
print()
for i, (title, file) in enumerate(sources, 1):
    print(f"[{i}] {title}  ({file})")

nums = []
for grp in re.findall(r"\[([\d,\s]+)\]", text):
    nums += [int(n) for n in re.findall(r"\d+", grp)]
print("ссылки:", sorted(set(nums)), "| фрагментов:", len(sources))
print("выдуманные номера:", [n for n in nums if n > len(sources)])

`

KeyboardInterrupt: 

In [48]:
queries = [
    "how does DataLoader num_workers work",
    "как сохранить и загрузить модель",
    "what is torch.nn.Linear",
    "how to use torch.magic_optimizer",
    "how to bake a chocolate cake",
    "как работает zero_grad"
]

for q in queries:
    text, sources = answer(q)
    print()
    for i, (title, file) in enumerate(sources, 1):
        print(f"[{i}] {title}  ({file})")
    
    nums = []
    for grp in re.findall(r"\[([\d,\s]+)\]", text):
        nums += [int(n) for n in re.findall(r"\d+", grp)]
    print("ссылки:", sorted(set(nums)), "| фрагментов:", len(sources))
    print("выдуманные номера:", [n for n in nums if n > len(sources)])

`num_workers` in `torch.utils.data.DataLoader` enables multi-process data loading. When set to a positive integer, it creates a specified number of worker processes to load data concurrently, which helps in reducing the time spent on data loading and allows the main process to perform other computations. Each worker process is initialized with the dataset, `collate_fn`, and `worker_init_fn` provided to the DataLoader. The `worker_init_fn` can be used to set up each worker process differently, such as seeding random number generators. However, there is a warning about potential issues with memory consumption and Python object references, which can be mitigated by using non-refcounted representations like Pandas, Numpy, or PyArrow objects. [1]
[1] torch.utils.data > Single- and Multi-process Data Loading > Multi-process data loading  (utils/data.md)
[2] Reproducibility > DataLoader  (notes/randomness.md)
[3] torch.utils.data.get_worker_info  (docstring)
[4] torch.utils.data > Single- and